In [1]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

login(userdata.get("HF_TOKEN"))

path = hf_hub_download("Exploration-Lab/CS-781-Capstone",
                        "Phase_1_Pretraining/Test/test.json",
                        repo_type="dataset")
with open(path, encoding="utf-8") as f:
    raw = f.read()
print(raw[:1000])



test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

{
  "1": {
    "text": "The issue involved in these petitions is as to whether the acquisition proceedings can be challenged at a belated stage. The issue is no more res integra as the issue has been considered by this Court time and again. 6. When a person challenges Section 4 Notification on any ground, it should be challenged within a reasonable period, and if the acquisition is challenged at a belated stage, the petition deserves to be dismissed only on this count. (Vide Hari Singh & Ors. Vs. State of U.P., AIR 1984 SC 1020). 7. A Constitution Bench of this Court, in [Aflatoon & Ors. Vs. Lt. Governor, Delhi & Ors.] AIR 1974 SC 2077, while dealing with the issue, observed as under:- \".... to have sat on the fence and allowed the government to complete the acquisition on the basis that notification under Section 4 and the declaration under Section 6 were valid and then to attack the notification on the grounds which were available to them at the time when the notification was publis

In [2]:
path = hf_hub_download("Exploration-Lab/CS-781-Capstone",
                        "Phase_1_Pretraining/Train/train_acts.json",
                        repo_type="dataset")
with open(path, encoding="utf-8") as f:
    head = f.read(2000)
print(head)


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

[
{"title": "Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996", "category": "Greater Bengaluru City Corporation - Act", "year": 1972, "url": "https://indiankanoon.org/doc/179573747/", "text": "Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors including ground floor. The Corporation of the City of Bangalore granted permission for construction of buildings beyond five floors in certain cases. In respect of some of the buildings the permission granted by the Corporation contrary to the Zonal Regulations then in force was questioned in the High Court and the Division Bench of the High Court held that the permission granted in violation of the Zonal Regulation was illegal. The matter was taken to the Supreme Court. 

In [3]:
for fname in ["train_supreme_court.json", "train_high_courts.json", "train_tribunals.json"]:
    from huggingface_hub import HfFileSystem
    fs = HfFileSystem()
    full_path = f"datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/{fname}"
    with fs.open(full_path, "rb") as f:
        head = f.read(1500)
    print(f"=== {fname} ===")
    print(head.decode("utf-8", errors="replace"))
    print()


=== train_supreme_court.json ===
[
{"title": "His Holiness Sri Vishwothama Thirtha ... vs The State Of Mysore", "date": "1961-12-18", "court": "Supreme Court of India", "citations": "AIR 1966 SUPREME COURT 1882, 1962 2 SCJ 131", "bench": "", "headnote": "The shrine in suit which was originally founded by Shri Madhavacharya, a Hindu saint was managed for a long time in rotation by the heads of eight Mutts which were also founded by the said saint for eight of his disciples. Large number of pilgrims from all over the country visited this shrine without any restraint, took part in the worship there, made offerings to the deity and received the prasad. The institution was managed with the monetary aid received from the State and contributions raised by the said heads of the eight Mutts from devotees resident in different parts of the country. The question arising for decision was whether the shrine in question was a \"temple\" within the meaning of s. 2(1) of the Madras Temple Entry Author

In [5]:
!pip install -q ijson
import ijson

full_path = "datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/train_supreme_court.json"
with fs.open(full_path, "rb") as f:
    first_record = next(ijson.items(f, "item"))

print("Supreme Court record keys:", list(first_record.keys()))
print("Has 'text' key:", "text" in first_record)
if "text" in first_record:
    print("text length (chars):", len(first_record["text"]))
    print("text preview:", first_record["text"][:300])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 4.6 MB/s eta 0:00:00
Supreme Court record keys: ['title', 'date', 'court', 'citations', 'bench', 'headnote', 'text', 'word_count', 'text_length_chars']
Has 'text' key: True
text length (chars): 24257
text preview: PETITIONER: HIS HOLINESS SRI VISHWOTHAMA THIRTHA SWAMIAR OFSODE MUTT Vs.

RESPONDENT: THE STATE OF MYSORE

DATE OF JUDGMENT: 18/12/1961

ACT: Temple-Large number of pilgrims visiting and worshiping-If private or public-Madras Temple Entry Authorisation Act, 1947(Mad. V. of 1947), s. 2(1).

HEADNOTE:


In [6]:
full_path = "datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/train_high_courts.json"
with fs.open(full_path, "rb") as f:
    records = []
    for i, rec in enumerate(ijson.items(f, "item")):
        records.append(rec.get("word_count", len(rec["text"].split())))
        if i >= 200:
            break

import statistics
print("sampled", len(records), "High Court records")
print("min:", min(records), "max:", max(records))
print("mean:", round(statistics.mean(records)), "median:", statistics.median(records))
print("first 20 word counts:", records[:20])


sampled 201 High Court records
min: 44 max: 6479
mean: 178 median: 95
first 20 word counts: [175, 84, 44, 192, 157, 96, 73, 95, 219, 74, 95, 103, 137, 152, 322, 159, 146, 113, 650, 70]


In [7]:
for src in ["train_acts.json", "train_tribunals.json"]:
    full_path = f"datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/{src}"
    with fs.open(full_path, "rb") as f:
        counts = []
        for i, rec in enumerate(ijson.items(f, "item")):
            counts.append(rec.get("word_count", len(rec["text"].split())))
            if i >= 200:
                break
    print(src, "-> sampled", len(counts), "min:", min(counts), "max:", max(counts),
          "mean:", round(statistics.mean(counts)), "median:", statistics.median(counts))


train_acts.json -> sampled 201 min: 117 max: 66781 mean: 5830 median: 2465
train_tribunals.json -> sampled 201 min: 69 max: 42119 mean: 4509 median: 3470
